# PDF → Markdown + Images + LaTeX — v4 (kernel-proof)

Kegagalan v3 bukan soal versi: pip sudah memasang marker 1.10.2 / surya 0.17.1 (terbaca di log kamu), tapi **kernel masih memegang modul surya 0.22 yang ter-import dari run sebelumnya** — Python tidak membuang modul di `sys.modules` saat pip mengganti file di disk. Buktinya traceback masih masuk `vllm.py` yang tidak ada di 0.17.1, dengan baris source yang tidak nyambung.

v4 menghilangkan seluruh kelas masalah itu: **marker dijalankan via CLI di subprocess** — interpreter baru setiap kali, selalu memakai versi di disk. Notebook tidak pernah meng-import marker/surya. Folder output juga di-wipe total tiap run, jadi sisa run lama tidak mengganggu.

Pakai: attach dataset PDF apa pun ke input, accelerator **GPU T4 x2**, Run All. Selesai.

In [1]:
# Install pinned. Kalau cell ini pernah jalan dengan versi lain di session ini,
# TIDAK masalah — pipeline memakai subprocess, bukan modul di kernel.
!pip install -q "marker-pdf==1.10.2" "surya-ocr==0.17.1" pymupdf4llm
import importlib.metadata as im, torch
print("marker:", im.version("marker-pdf"), "| surya:", im.version("surya-ocr"))
print("torch :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.9/189.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [2]:
"""
PDF -> Markdown + images/ + LaTeX (Kaggle T4). v4: kernel-proof.
Marker dijalankan via CLI subprocess (interpreter segar) — kebal stale kernel.
Tidak ada import marker/surya di kernel notebook.
Output: /kaggle/working/output/<slug>/{<slug>.md, images/, <slug>_meta.json}
"""

import json, os, re, shutil, subprocess, sys, time, unicodedata
from pathlib import Path

# ----------------------------------------------------------------- CONFIG
INPUT_DIR  = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/output")
STAGING    = Path("/kaggle/working/_staging")
MARKER_RAW = Path("/kaggle/working/_marker_raw")
FORCE_OCR  = True
# --------------------------------------------------------------------------

# ---- sanity: versi di DISK (bukan modul ter-load) --------------------------
import importlib.metadata as im
mv, sv = im.version("marker-pdf"), im.version("surya-ocr")
print(f"disk: marker-pdf {mv} | surya-ocr {sv}")
assert mv == "1.10.2" and sv == "0.17.1", \
    "Versi salah. Jalankan ulang cell install (pin 1.10.2 / 0.17.1)."

import torch
assert torch.cuda.is_available(), "GPU tidak aktif. Set accelerator T4 x2."
cap = torch.cuda.get_device_capability(0)
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} tak didukung torch ini. Pakai T4, bukan P100."
print("GPU:", torch.cuda.get_device_name(0), f"(sm_{cap[0]}{cap[1]})")


def slugify(stem: str) -> str:
    s = unicodedata.normalize("NFKD", stem).encode("ascii", "ignore").decode()
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")
    return s[:80] or "doc"


def qc_report(md: str, n_images: int, src: str, engine: str, elapsed: float) -> dict:
    headings = re.findall(r"^#{1,6}\s+(.+)$", md, flags=re.M)
    return {
        "source_pdf": src, "engine": engine, "elapsed_sec": round(elapsed, 1),
        "chars": len(md),
        "headings": len(headings), "heading_titles": headings[:30],
        "display_equations": len(re.findall(r"\$\$.+?\$\$", md, flags=re.S)),
        "inline_equations": len(re.findall(r"(?<!\$)\$(?!\$)[^$\n]+?\$(?!\$)", md)),
        "markdown_table_rows": len(re.findall(r"^\|.+\|\s*$", md, flags=re.M)),
        "html_tables": md.lower().count("<table"),
        "images_saved": n_images,
        "references_section_found": bool(re.search(
            r"^#{1,6}\s*\**\s*(references|daftar pustaka|bibliography|literature cited)",
            md, flags=re.M | re.I)),
    }


# ---- STAGE 0: bersih total + staging slug -----------------------------------
for d in (OUTPUT_DIR, STAGING, MARKER_RAW):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)

pdfs = sorted(p for p in INPUT_DIR.rglob("*.pdf") if p.stat().st_size > 0)
if not pdfs:
    raise SystemExit(f"Tidak ada PDF di {INPUT_DIR}")

slug_to_src = {}
for p in pdfs:
    slug = slugify(p.stem)
    base, i = slug, 2
    while slug in slug_to_src:                      # hindari tabrakan nama
        slug = f"{base}_{i}"; i += 1
    slug_to_src[slug] = p
    shutil.copy2(p, STAGING / f"{slug}.pdf")
    print(f"staged: {p.name} -> {slug}.pdf")

# ---- STAGE 1: marker batch CLI di SUBPROCESS --------------------------------
env = dict(os.environ, TORCH_DEVICE="cuda")
cmd = ["marker", str(STAGING),
       "--output_dir", str(MARKER_RAW),
       "--output_format", "markdown",
       "--workers", "1"]
if FORCE_OCR:
    cmd.append("--force_ocr")
print("\n[marker-cli]", " ".join(cmd))
t0 = time.time()
proc = subprocess.run(cmd, env=env, text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
marker_elapsed = time.time() - t0
print(proc.stdout[-4000:])
print(f"[marker-cli] exit={proc.returncode} elapsed={marker_elapsed:.0f}s")

# ---- STAGE 2: post-process hasil marker -> struktur final --------------------
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".gif"}

def collect_marker_result(slug: str):
    """Cari folder hasil marker untuk slug ini; None kalau tak ada .md."""
    cand = MARKER_RAW / slug
    if cand.is_dir() and list(cand.glob("*.md")):
        return cand
    for d in MARKER_RAW.rglob("*"):
        if d.is_dir() and d.name == slug and list(d.glob("*.md")):
            return d
    return None

def finalize_marker(slug: str, raw_dir: Path, out_dir: Path, per_doc_sec: float):
    md_file = next(raw_dir.glob("*.md"))
    md = md_file.read_text(encoding="utf-8")
    img_dir = out_dir / "images"; img_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in raw_dir.iterdir():
        if f.suffix.lower() in IMG_EXT:
            shutil.copy2(f, img_dir / f.name)
            md = md.replace(f"]({f.name})", f"](images/{f.name})")
            n += 1
    (out_dir / f"{slug}.md").write_text(md, encoding="utf-8")
    return qc_report(md, n, str(slug_to_src[slug]), "marker", per_doc_sec)

FALLBACK_SCRIPT = r"""
import sys, pathlib, pymupdf4llm
staged, out_dir = sys.argv[1], pathlib.Path(sys.argv[2])
img_dir = out_dir / "images"; img_dir.mkdir(parents=True, exist_ok=True)
md = pymupdf4llm.to_markdown(staged, write_images=True, image_path=str(img_dir),
                             image_format="png", table_strategy="lines_strict")
md = md.replace(str(img_dir) + "/", "images/")
stem = pathlib.Path(staged).stem
(out_dir / f"{stem}.md").write_text(md, encoding="utf-8")
"""
fb_path = Path("/kaggle/working/_fb.py"); fb_path.write_text(FALLBACK_SCRIPT)

def run_fallback(slug: str, out_dir: Path) -> dict:
    staged = STAGING / f"{slug}.pdf"
    t0 = time.time()
    r = subprocess.run([sys.executable, str(fb_path), str(staged), str(out_dir)],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        return {"source_pdf": str(slug_to_src[slug]), "engine": "FAILED"}
    md = (out_dir / f"{slug}.md").read_text(encoding="utf-8")
    n = len(list((out_dir / "images").glob("*")))
    meta = qc_report(md, n, str(slug_to_src[slug]), "pymupdf4llm-fallback",
                     time.time() - t0)
    meta["warning"] = "Equation TIDAK dalam LaTeX pada mode fallback."
    return meta

all_meta = []
per_doc = marker_elapsed / max(len(slug_to_src), 1)
for slug in slug_to_src:
    out_dir = OUTPUT_DIR / slug; out_dir.mkdir(parents=True, exist_ok=True)
    raw = collect_marker_result(slug)
    if raw is not None:
        meta = finalize_marker(slug, raw, out_dir, per_doc)
    else:
        print(f"[{slug}] hasil marker tidak ditemukan -> fallback")
        meta = run_fallback(slug, out_dir)
    (out_dir / f"{slug}_meta.json").write_text(json.dumps(meta, indent=2))
    all_meta.append(meta)
    print(json.dumps({k: v for k, v in meta.items() if k != "heading_titles"},
                     indent=2))

(OUTPUT_DIR / "summary.json").write_text(json.dumps(all_meta, indent=2))
engines = [m.get("engine") for m in all_meta]
print("\nEngine per file:", engines)
if any(e != "marker" for e in engines):
    print("PERINGATAN: ada file yang tidak lewat marker. Baca log CLI di atas.")
print(f"Selesai. Output di {OUTPUT_DIR}")

disk: marker-pdf 1.10.2 | surya-ocr 0.17.1
GPU: Tesla T4 (sm_75)
staged: FLBP-Keramidas.pdf -> FLBP-Keramidas.pdf
staged: GLCM-Sebastian.pdf -> GLCM-Sebastian.pdf
staged: Introduction to Local Binary Patterns New Variants and Applications.pdf -> Introduction_to_Local_Binary_Patterns_New_Variants_and_Applications.pdf
staged: LBP-Ojala.pdf -> LBP-Ojala.pdf

[marker-cli] marker /kaggle/working/_staging --output_dir /kaggle/working/_marker_raw --output_format markdown --workers 1 --force_ocr
%|          | 0.00/858 [00:00<?, ?B/s]













































Processing PDFs: 100%|██████████| 4/4 [18:55<00:00, 283.75s/pdf]
2026-08-15 14:25:54,437 [INFO] marker: Stopped NVIDIA MPS server for chunk 0
INFO:marker:Stopped NVIDIA MPS server for chunk 0
Inferenced 85 pages in 1135.33 seconds, for a throughput of 0.07 pages/sec for chunk 1/1

[marker-cli] exit=0 elapsed=1187s
{
  "source_pdf": "/kaggle/input/datasets/daffaelgosantosa/pdf-dataset/FLBP-Keramidas.pdf",
  "engine": "ma

## Inspeksi hasil

In [3]:
import json, pathlib
OUT = pathlib.Path("/kaggle/working/output")
for d in sorted(OUT.iterdir()):
    if not d.is_dir():
        continue
    mfile = d / f"{d.name}_meta.json"
    if not mfile.exists():
        print(f"[skip] {d.name}: tidak ada meta"); continue
    meta = json.loads(mfile.read_text())
    print("="*70)
    for k in ["source_pdf","engine","elapsed_sec","headings",
              "display_equations","inline_equations",
              "markdown_table_rows","images_saved",
              "references_section_found"]:
        print(f"{k:28s}: {meta.get(k)}")

source_pdf                  : /kaggle/input/datasets/daffaelgosantosa/pdf-dataset/FLBP-Keramidas.pdf
engine                      : marker
elapsed_sec                 : 296.7
headings                    : 18
display_equations           : 12
inline_equations            : 81
markdown_table_rows         : 68
images_saved                : 5
references_section_found    : True
source_pdf                  : /kaggle/input/datasets/daffaelgosantosa/pdf-dataset/GLCM-Sebastian.pdf
engine                      : marker
elapsed_sec                 : 296.7
headings                    : 16
display_equations           : 5
inline_equations            : 50
markdown_table_rows         : 0
images_saved                : 7
references_section_found    : True
source_pdf                  : /kaggle/input/datasets/daffaelgosantosa/pdf-dataset/Introduction to Local Binary Patterns New Variants and Applications.pdf
engine                      : marker
elapsed_sec                 : 296.7
headings                    :

In [4]:
# Sampel equation, tabel, gambar — ganti idx untuk file lain
import pathlib, re
idx = 0
dirs = sorted(p for p in pathlib.Path("/kaggle/working/output").iterdir() if p.is_dir())
d = dirs[idx]
print("dokumen:", d.name)
md_text = next(d.glob("*.md")).read_text()

eqs = re.findall(r"\$\$(.+?)\$\$", md_text, flags=re.S)
print(f"--- {len(eqs)} display equations, 3 pertama ---")
for e in eqs[:3]:
    print(e.strip()[:200], "\n")

rows = [l for l in md_text.splitlines() if l.startswith("|")]
print(f"--- {len(rows)} baris tabel, 8 pertama ---")
print("\n".join(rows[:8]))

imgs = sorted((d/"images").glob("*"))
print(f"--- {len(imgs)} gambar ---")
for i in imgs[:10]:
    print(i.name)

dokumen: FLBP-Keramidas
--- 12 display equations, 3 pertama ---
B = \{ p_x \mid L_B(x) \} \tag{1} 

m_B(x) = \begin{cases} 1 & \text{if } p_x \in B \\ 0 & \text{if } p_x \notin B \end{cases} 

BP_{CODE} = \sum_{x=0}^{k-1} d_x \cdot w_x \tag{3} 

--- 68 baris tabel, 8 pertama ---
| Application        | Reported in |
|--------------------|-------------|
| Visual Inspection  | [17-19]     |
| Remote Sensing     | [20]        |
| Image & Video      | [21-23]     |
| Face Recognition   | [24-27]     |
| Biomedical Imaging | [28-30]     |
| Motion Analysis    | [31-33]     |
--- 5 gambar ---
_page_12_Figure_2.jpeg
_page_2_Figure_14.jpeg
_page_4_Figure_2.jpeg
_page_5_Figure_2.jpeg
_page_7_Figure_5.jpeg


In [5]:
# Zip hasil untuk diunduh dari panel Output
import shutil
shutil.make_archive("/kaggle/working/pdf2md_output", "zip", "/kaggle/working/output")
print("done: /kaggle/working/pdf2md_output.zip")

done: /kaggle/working/pdf2md_output.zip


## Verifikasi
- Baris `Engine per file:` di akhir cell pipeline harus `['marker', 'marker', 'marker', 'marker']`.
- FLBP-Keramidas: ±11 display equation `$$...$$`, baris tabel Table 1–9, gambar Figure utuh di `images/`, `references_section_found: true`.
- Run pertama mengunduh model Surya (~3GB); run berikutnya dalam session yang sama langsung jalan.